### **Token-Level Chunking*

In [11]:
import tiktoken
from pypdf import PdfReader
from langchain_text_splitters import TokenTextSplitter
from langchain_ollama import OllamaEmbeddings

In [ ]:
# pdf_path = "./docs/PAPER1.pdf"
# reader = PdfReader(pdf_path)
# text = ""
# for page in reader.pages:
#     text += page.extract_text() + "\n"

# print(f"PDF loaded. Total characters: {len(text)}")

PDF loaded. Total characters: 39602


In [ ]:
## - manually coded

# encoding = tiktoken.get_encoding("cl100k_base")  # adjust for your model
# tokens = encoding.encode(text)
# token_chunks = []
# chunk_size = 512
# overlap = 50
# for i in range(0, len(tokens), chunk_size - overlap):
#     token_chunks.append(encoding.decode(tokens[i:i+chunk_size]))
# print(f"Token-level chunks: {len(token_chunks)}")

In [ ]:
# Split all docs into token chunks

# token_chunks = token_splitter.split_documents(text)

# print(f"Total token chunks: {len(token_chunks)}")

# # Print a sample

# print("Sample chunk:", token_chunks[0].page_content[:300], "...")

In [12]:
## - using framework

from langchain_community.document_loaders import PyPDFLoader
pdf_path = "./docs/PAPER1.pdf"
loader = PyPDFLoader(pdf_path)

# Load the PDF into a list of LangChain Document objects
docs = loader.load()

print(f"Loaded {len(docs)} page(s) from the PDF")

Loaded 15 page(s) from the PDF


In [13]:
## - using framework

# chunk_size = token count you want per chunk
# chunk_overlap = token overlap between chunks
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",  # use tokenizer matching your LLM/embedding model
    chunk_size=500,               # 500 tokens per chunk
    chunk_overlap=50              # 50 token overlap
)

In [14]:
splitted_docs = token_splitter.split_documents(docs)

splitted_docs[0].page_content[:400]

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nG'

In [ ]:
## - Now you can embed this chunked docs using any embedding algo.


---

## 🧠 Why TokenTextSplitter?

Unlike character or sentence splitters, **TokenTextSplitter** counts tokens using the model’s encoding, so your chunks reflect the exact token limits you care about. ([LangChain Docs][1])

---

## ✅ Code Example

```python
# install required libraries if you haven't already
# pip install langchain-community tiktoken

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitters import TokenTextSplitter

# -----------------------------
# 1. Load the PDF as Documents
# -----------------------------
pdf_path = "./docs/PAPER1.pdf"
loader = PyPDFLoader(pdf_path)

# Load the PDF into a list of LangChain Document objects
docs = loader.load()

print(f"Loaded {len(docs)} page(s) from the PDF")

# -----------------------------
# 2. Token‑based splitting
# -----------------------------
# chunk_size = token count you want per chunk
# chunk_overlap = token overlap between chunks
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",  # use tokenizer matching your LLM/embedding model
    chunk_size=500,               # 500 tokens per chunk
    chunk_overlap=50              # 50 token overlap
)

# Split all docs into token chunks
token_chunks = token_splitter.split_documents(docs)

print(f"Total token chunks: {len(token_chunks)}")

# Print a sample
print("Sample chunk:", token_chunks[0].page_content[:300], "...")
```

---

## 📌 Breakdown of key lines

### 📄 Loading PDF

```python
docs = loader.load()
```

* Returns a list of **Documento** objects with `.page_content` and metadata. ([LangChain Docs][2])

---

### 🔗 Configuring TokenTextSplitter

```python
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)
```

* **encoding_name="cl100k_base"** uses the same token encoding as many OpenAI models. ([LangChain Docs][1])
* **chunk_size** is the **max tokens per chunk**
* **chunk_overlap** adds context between chunks — useful in retrieval

---

### 📦 Splitting into chunks

```python
token_chunks = token_splitter.split_documents(docs)
```

* This returns a list of split `Document` objects where each chunk respects token constraints.

---

## 📌 When to use this

Use token splitting when:

* you care about **token limits for embeddings** or LLM context windows
* you want chunks that won’t be truncated by the model
* character lengths vary drastically

Character‑based splitting *may* split in the wrong place because token count and character count don’t align. ([LangChain Docs][1])

---

## 🚀 Tip for RAG

After splitting:

1. Compute embeddings for `token_chunks`
2. Upsert into your vector store
3. Use vector search → LLM generation

This ensures:

* consistent chunk sizes
* coherent context
* minimal token overflow

---



[1]: https://docs.langchain.com/oss/python/integrations/splitters/split_by_token?utm_source=chatgpt.com "Splitting by token - Text splitter integration guide - Docs by LangChain"
[2]: https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader/?utm_source=chatgpt.com "PyPDFLoader - Docs by LangChain"
